## ___Updating the mycorrhizal states___
--------------------

In [1]:
!python --version

Python 3.13.7


In [7]:
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd
from bs4 import BeautifulSoup

In [4]:
# https://datadryad.org/dataset/doi:10.5061/dryad.n8bm9
maherali_crude = pd.read_excel(r"../../data/chapter2/Maherali.etal.AmNat.Data.xlsx", sheet_name="Original states data", skiprows=range(2))
maherali_crude.rename(mapper={old: old.replace('(', '').replace(')', '').lower().replace(' ', '_') for old in maherali_crude.columns}, axis=1, inplace=True) # column names have parentheses and spaces
# taxonomy columns in Maherali et. al. dataset has trailing spaces :(
maherali_crude.loc[:, "original_name_genus_species"] = maherali_crude.original_name_genus_species.str.strip()
maherali_crude.loc[:, "raw_state_record_from_publication"] = maherali_crude.raw_state_record_from_publication.str.strip()

final_maherali = pd.read_excel(r"../../data/chapter2/Maherali.etal.AmNat.Data.xlsx", sheet_name="Final list matched with phylo", skiprows=range(2))
final_maherali.rename(mapper={old: old.lower().replace(' ', '_') for old in final_maherali.columns}, axis=1, inplace=True)
final_maherali.genus_species = final_maherali.genus_species.str.strip().str.replace('_', ' ') # the sheet "Final list matched with phylo" has genus and specific epithets concatenated by under scores!

# in TRY, mycorrhiza type is trait id 7
try_myco = pd.read_csv(r"../../data/chapter2/TRY/mycorrhizal_states.txt", delimiter='\t', low_memory=False, encoding="latin1", usecols=["Dataset", "SpeciesName", "AccSpeciesName", "OrigValueStr",
                                "TraitID"]).dropna(subset=["AccSpeciesName", "OrigValueStr", "TraitID"])
# unify the mycorrhizal state info
# 'ECTO', 'NM/AM', 'EC', 'EC/AM', 'AM', 'Ecto', 'Non',        'vesicular-arbuscular mycorrhiza', 'ectomycorrhiza', 'no', '0', 'Ph.th.end.', 'VAM', 'Ectomycorrhiza', 'E.ch.ect.', 'arbuscular',
# 'ec?', 'VA', 'ecto', 'Absent', 'non-ectomycorrhizal', 'ectomycorrhizal', 'Yes', 'No', 'EM', 'AMNM', 'NM', 'AM + EM', 'ERM', 'Ericoid', 'ECM'

MYCORRHIZAL_STATES_REPLACEMENTS = {
    "ECTO": "EM",
    "Ecto": "EM",
    "EC": "EM",
    "ectomycorrhiza": "EM",
    "Ectomycorrhiza": "EM",
    "ecto": "EM",
    "ectomycorrhizal": "EM",
    "ECM": "EM",
    "vesicular-arbuscular mycorrhiza" : "AM",
    "VAM": "AM",
    "VA": "AM",
    "Non": "NM",
    "AMNM": "NM/AM",
    "Ericoid": "ER",
    "ERM": "ER",
    "AM + EM": "AM/EM",
    "EC/AM": "AM/EM"
}
try_myco.loc[:, "OrigValueStr"] = try_myco.OrigValueStr.replace(MYCORRHIZAL_STATES_REPLACEMENTS)

mycodb_v4 = pd.read_csv(r"../../data/chapter2/MycoDB_version4.csv", usecols=["PlantSpecies2018", "FUNGROUP", "MYCORRHIZAETYPE", "AM_single_genus", "EM_single_genus", 
                        "STERILIZED", "NONMYCOCONTROL", "NONMYCOCONTROL2"]).dropna(subset="PlantSpecies2018").drop_duplicates()
mycodb_v4.loc[:, "PlantSpecies2018"] = mycodb_v4.PlantSpecies2018.str.capitalize().str.replace('_', ' ')

# scrape the online only MycoDB metadata and serialize it to the disk
# req = Request(url=r"https://www.nature.com/articles/sdata201628/tables/2", headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:142.0) Gecko/20100101 Firefox/142.0"})
# with urlopen(req) as r:
#     soup = BeautifulSoup(r.read())
# 
# table = soup.find(name="table", attrs={"class": "data last-table"}) # locate the metadata table
# [th.text.strip() for th in table.find_all(name="th")] # column names
# mycodb_descriptions = [[td.text for td in tr.find_all(name="td")] for tr in table.find_all(name="tr")[1:]] # parse the rows
# 
# # create a dataframe using the parsed rows and column names and serialize it to the disk
# pd.DataFrame({ 
#     "Variable": [row[0] for row in mycodb_descriptions],
#     "Description": [row[1] for row in mycodb_descriptions],
#     "Variable Type (range)": [row[2] for row in mycodb_descriptions],
#     "Levels (#studies/level)": [row[3] for row in mycodb_descriptions],
# }).to_csv(r"../data/chapter2/MycoDB_version4_metadata.csv", index=False)

mycodb_v4_meta = pd.read_csv(r"../../data/chapter2/MycoDB_version4_metadata.csv")

subset_categorical = pd.read_csv(r"../../data/chapter2/FREDv3subset/FRED_subset_categorical.csv")

In [5]:
subset_categorical

,binominal,F01286,F01287,F01289,F01290,F00043,F00645,F00004
0,Populus trichocarpa,Populus,trichocarpa,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord..."
1,Populus tremula,Populus,tremula,Salicaceae,Malpighiales,C3,NaN,"Hajek P, Hertel D, Leuschner C. 2014. Root ord..."
2,Altingia obovata,Altingia,obovata,Altingiaceae,Saxifragales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi..."
3,Cryptocarya chinensis,Cryptocarya,chinensis,Lauraceae,Laurales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi..."
4,Elaeocarpus sylvestris,Elaeocarpus,sylvestris,Elaeocarpaceae,Oxalidales,C3,NaN,"Xu Y, Gu JC, Dong XY, Liu Y, Wang ZQ. 2011. Fi..."
...,...,...,...,...,...,...,...,...
236,Populus deltoides,Populus,deltoides,Salicaceae,Malpighiales,C3,NaN,Valverde et al (unpublished)
237,Prunus sargentii,Prunus,sargentii,Rosaceae,Rosales,C3,NaN,Valverde et al (unpublished)
238,Styphnolobium japonicum,Styphnolobium,japonicum,Fabaceae,Fabales,C3,NaN,Valverde et al (unpublished)
239,Syringa reticulata,Syringa,reticulata,Oleaceae,Lamiales,C3,NaN,Valverde et al (unpublished)
